In [4]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "sqlalchemy"], check=True)
print("Dependencies installed!")

Dependencies installed!


# 05 — Staging Verification & Summary
Connects to PostgreSQL and verifies that all three staging tables
were loaded correctly. Covers row counts, null checks, sample records
and a final pipeline summary report.

In [5]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

print("Connected!")

Connected!


## Staging Row Counts vs Raw Row Counts

In [6]:
query = """
SELECT
    'customers' AS table_name,
    (SELECT COUNT(*) FROM raw.customers)     AS raw_count,
    (SELECT COUNT(*) FROM staging.customers) AS staging_count
UNION ALL
SELECT
    'products',
    (SELECT COUNT(*) FROM raw.products),
    (SELECT COUNT(*) FROM staging.products)
UNION ALL
SELECT
    'orders',
    (SELECT COUNT(*) FROM raw.orders),
    (SELECT COUNT(*) FROM staging.orders);
"""

df_counts = pd.read_sql(query, engine)
df_counts["match"] = df_counts["raw_count"] == df_counts["staging_count"]
df_counts

,table_name,raw_count,staging_count,match
0,customers,282,282,True
1,products,88,88,True
2,orders,37891,37891,True


## Staging Null Checks

In [7]:
print("=== staging.customers ===")
df_c = pd.read_sql("SELECT * FROM staging.customers;", engine)
print(df_c.isnull().sum().to_string())

print("\n=== staging.products ===")
df_p = pd.read_sql("SELECT * FROM staging.products;", engine)
print(df_p.isnull().sum().to_string())

print("\n=== staging.orders ===")
df_o = pd.read_sql("SELECT * FROM staging.orders;", engine)
print(df_o.isnull().sum().to_string())

=== staging.customers ===
customer_id    0
name           0
email          0
phone          0
city           0
batch_id       0
created_at     0
loaded_at      0

=== staging.products ===
product_id      0
product_name    0
category        0
price           0
batch_id        0
created_at      0
loaded_at       0

=== staging.orders ===
order_id           0
customer_id        0
product_id         0
quantity           0
unit_price         0
total_price        0
status             0
payment_method     0
shipping_method    0
batch_id           0
created_at         0
loaded_at          0


## Pipeline Summary Report

In [8]:
from datetime import datetime, timezone

print("=" * 50)
print("  MANDERA_ANALYTICS PIPELINE SUMMARY")
print("=" * 50)
print(f"  Report time : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
print()
print("  LAYER        TABLE        ROWS")
print("  ---------    ---------    -------")
print(f"  Raw          customers    {len(df_c):,}")
print(f"  Raw          products     {len(df_p):,}")
print(f"  Raw          orders       {len(df_o):,}")
print()
print(f"  Staging      customers    {len(df_c):,}")
print(f"  Staging      products     {len(df_p):,}")
print(f"  Staging      orders       {len(df_o):,}")
print()
print("  STATUS: ALL TABLES LOADED SUCCESSFULLY")
print("=" * 50)

  MANDERA_ANALYTICS PIPELINE SUMMARY
  Report time : 2026-04-22 18:21:54 UTC

  LAYER        TABLE        ROWS
  ---------    ---------    -------
  Raw          customers    282
  Raw          products     88
  Raw          orders       37,891

  Staging      customers    282
  Staging      products     88
  Staging      orders       37,891

  STATUS: ALL TABLES LOADED SUCCESSFULLY
